# Basic Measurement Layout

In [ ]:
!pip install pymc --quiet
!pip install arviz --quiet

In [ ]:
import arviz as az
import graphviz
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import random as rm
import seaborn as sns

from IPython.display import Image
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from pymc import model

print(f"Running on PyMC v{pm.__version__}")

In [ ]:
import pandas as pd
import requests
from io import StringIO

# Base URL for raw GitHub files
base_url = "https://raw.githubusercontent.com/Kinds-of-Intelligence-CFI/measurement-layouts/refs/heads/main/basic_measurement_layout/data/"

# List your CSV file names
pixel_sizes = range(4, 44, 4)
noise_levels = [0.0, 0.2, 0.4, 0.6, 0.8]

# Read and combine
dataframes = []
for p in pixel_sizes:
  for n in noise_levels:
    url = base_url + f"results_pixels_{p}_noise_{n}.csv"
    response = requests.get(url)
    df = pd.read_csv(StringIO(response.text))
    dataframes.append(df)

# Combine all DataFrames
combined_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
def logistic_general(x, min, max, c = 0, p = 0.99):

  """
  Generalized version of the logistic where min can be any number, not just 0 as above.

  :param min: The min ability/demand
  :param max: The max ability/demand
  :param c: Set different from 0 when we are dealing with multiple choice responses; e.g. c is 0.5 when the response is yes/no, and c is 0.25 if there are 4 possible choices
  :param p: The probability we want to set the max margin to (e.g. 0.99 or 0.999); the min margin will be set to 1 - p
  :param k: The slope of the logistic
  """
  max_n = max - min
  k = - np.log((1 - p )/(p - c)) / max_n
  return c + ((1 - c) / (1 + np.exp(-k * x)))

def brierDecomp(preds, outs):

  brier= 1/len(preds) * sum( (preds-outs)**2 )
  ## bin predictions
  bins = np.linspace(0,1,11)
  binCenters = (bins[:-1] +bins[1:]) /2
  binPredInds = np.digitize(preds,binCenters)
  binnedPreds = bins[binPredInds]

  binTrueFreqs = np.zeros(10)
  binPredFreqs = np.zeros(10)
  binCounts = np.zeros(10)

  for i in range(10):
      idx = (preds >= bins[i]) & (preds < bins[i+1])

      binTrueFreqs[i] = np.sum(outs[idx])/np.sum(idx) if np.sum(idx) > 0 else 0
     # print(np.sum(outs[idx]), np.sum(idx), binTrueFreqs[i])
      binPredFreqs[i] = np.mean(preds[idx]) if np.sum(idx) > 0 else 0
      binCounts[i] = np.sum(idx)

  calibration = np.sum(binCounts * (binTrueFreqs - binPredFreqs) ** 2) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  refinement = np.sum(binCounts * (binTrueFreqs *(1 - binTrueFreqs))) / np.sum(binCounts) if np.sum(binCounts) > 0 else 0
  # Compute refinement component
  #refinement = brier - calibration
  return brier, calibration,

def predict(m, trace, relevantData):
  with m:
    predictions = pm.sample_posterior_predictive(trace, var_names=["successP"], return_inferencedata=False, predictions=True, extend_inferencedata=False)
    predictionSuccessChainRuns = predictions["successP"][:,:,0:len(relevantData)]
    predictionsSuccessInstance = np.mean(predictionSuccessChainRuns, (0,1))
    successes = np.array([0 if val < -0.99 else 1 for val in data['finalReward']])

    return predictionsSuccessInstance, successes

In [ ]:
def setupModel(data):
  successes = [0 if val < -0.99 else 1 for val in data['finalReward']]

  abilityMin = {}
  abilityMax = {}

  abilityMin["navAbility"] = 0
  abilityMax["navAbility"] = data['distance'].max()

  abilityMin["visualAcuity"] = 0
  abilityMax["visualAcuity"] = (data['distance']/data['size']).min()

  m = pm.Model()
  with m:
    navAbility = pm.HalfNormal("navAbility", sigma = abilityMax["navAbility"]/2)
    visualAcuity = pm.HalfNormal("visualAcuity", sigma = abilityMax["visualAcuity"]/2)

    goalDist = pm.MutableData("goalDistance", data["distance"])
    goalSize = pm.MutableData("goalSize", data["size"])

    navP = pm.Deterministic("navP", logistic_general(navAbility - goalDist, min = abilityMin["navAbility"], max = abilityMax["navAbility"], c = 0, p = 0.99))
    visualP = pm.Deterministic("visualP", logistic_general(visualAcuity - goalSize, min = abilityMin["visualAcuity"], max = abilityMax["visualAcuity"], c = 0, p = 0.99))

    successP = pm.Deterministic("successP", navP * visualP)

    taskSuccess = pm.Bernoulli("taskSuccess", successP, observed = successes)

  return m, abilityMin, abilityMax, successes

In [ ]:
data = combined_df[(combined_df['pixelInput'] == 4) & (combined_df['navigationNoise'] == 0.0)]
m, abilityMin, abilityMax, successes = setupModel(data=data)
gv = pm.model_graph.model_to_graphviz(m)
gv

In [ ]:
pymc_sample_num = 2000

chains = 4

In [ ]:
from collections import defaultdict

results = defaultdict(list)

In [ ]:
for p in pixel_sizes:
  for n in noise_levels:
    results['pixels'].append(p)
    results['noise'].append(n)

    data = combined_df[(combined_df['pixelInput'] == p) & (combined_df['navigationNoise'] == n)]
    train_set, test_set = train_test_split(data, test_size = 0.2, random_state = 2024)

    model_train, _, _, _ = setupModel(data=train_set)

    with model_train:
      data_training = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains)

    model_test, _, _, _ = setupModel(data=test_set)

    predictionsSuccessInstance, successes = predict(model_test, data_training, test_set)

    agentBrierScoreSuccess, _, _ = brierDecomp(predictionsSuccessInstance, successes)
    agentAggBrierScoreSuccess, _, _ = brierDecomp(np.repeat(np.mean(successes), len(successes)), successes)

    results['modelBrier'].append(agentBrierScoreSuccess)
    results['aggBrier'].append(agentAggBrierScoreSuccess)

    model_all, abilityMin, abilityMax, successes = setupModel(data=data)

    with model_train:
      data_all = pm.sample(pymc_sample_num, target_accept=0.95, chains = chains)

    navMean = float(np.mean(data_all['posterior']['navAbility']))
    navStd = float(np.std(data_all['posterior']['navAbility']))
    visualMean = float(np.mean(data_all['posterior']['visualAcuity']))
    visualStd = float(np.std(data_all['posterior']['visualAcuity']))

    results['navigationMean'].append(navMean)
    results['navigationStd'].append(navStd)
    results['visualAcuityMean'].append(visualMean)
    results['visualAcuityStd'].append(visualStd)

    results['meanSuccess'].append(np.mean(successes))

